# GPT-2 Large: Accessibility Domain Knowledge

36 layers | 20 heads | d_model=1280 | d_mlp=5120 | sequential attn+MLP

In [1]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [2]:
model_name = "gpt2-large"
model, info = load_model(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-large into HookedTransformer
Loaded gpt2-large on mps
  36 layers | 20 heads | d_model=1280 | d_mlp=5120 | sequential attn+MLP


In [18]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
    ("A screen reader is", " software", "screenReaderv1"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: gpt2-large
Prompt: "The W in WCAG stands for"
Target: " Web" (token 5313)
Final prediction: " Women"

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         the                 2265           0.000050    
1         the                 3435           0.000048    
2         example             3074           0.000059    
3         example             2972           0.000060    
4         example             2940           0.000061    
5         sale                2258           0.000075    
6         example             1433           0.000116    
7         example             2071           0.000085    
8         example             1550           0.000100    
9         example             3126           0.000044    
10        symbol              4166           0.000032    
11        symbol              9349           0.000010    
12        symbol              11456 

In [19]:
for label, data in results.items():
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [20]:
for label, data in results.items():
    data["lens"].to_csv(f"../../results/gpt2/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/gpt2/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/gpt2/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/gpt2/{model_name}/figures")

Saved: ../../results/gpt2/gpt2-large/figures/wcag-logit-lens.png
Saved: ../../results/gpt2/gpt2-large/figures/wcag-decomposition.png
Saved: ../../results/gpt2/gpt2-large/figures/wcag-head-heatmap.png
Saved: ../../results/gpt2/gpt2-large/figures/aria-logit-lens.png
Saved: ../../results/gpt2/gpt2-large/figures/aria-decomposition.png
Saved: ../../results/gpt2/gpt2-large/figures/aria-head-heatmap.png
Saved: ../../results/gpt2/gpt2-large/figures/alt-logit-lens.png
Saved: ../../results/gpt2/gpt2-large/figures/alt-decomposition.png
Saved: ../../results/gpt2/gpt2-large/figures/alt-head-heatmap.png
Saved: ../../results/gpt2/gpt2-large/figures/HTML-logit-lens.png
Saved: ../../results/gpt2/gpt2-large/figures/HTML-decomposition.png
Saved: ../../results/gpt2/gpt2-large/figures/HTML-head-heatmap.png
Saved: ../../results/gpt2/gpt2-large/figures/screenReader-logit-lens.png
Saved: ../../results/gpt2/gpt2-large/figures/screenReader-decomposition.png
Saved: ../../results/gpt2/gpt2-large/figures/screenRea

In [6]:
unload(model)

Model unloaded, memory cleared
